In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F


@dp.materialized_view(
    name="bronze_players",
    comment="Raw player data from FPL API",
)
def bronze_players():
    return spark.read.format("json").load("/Volumes/fpl_project/bronze/raw_files/players.json")


@dp.materialized_view(
    name="silver_players",
    comment="Cleaned and typed player data",
)
@dp.expect("valid_player_id", "player_id IS NOT NULL")
@dp.expect("positive_price", "current_price > 0")
@dp.expect_or_drop("has_minutes", "minutes >= 0")
def silver_players():
    return (
        spark.read.table("bronze_players")
        .select(
            F.col("id").alias("player_id"),
            F.col("web_name").alias("display_name"),
            (F.col("now_cost") / 10).alias("current_price"),
            F.col("total_points").cast("int"),
            F.col("minutes").cast("int"),
            F.col("goals_scored").cast("int"),
            F.col("assists").cast("int"),
            F.col("expected_goals").cast("double").alias("xG"),
            F.col("expected_assists").cast("double").alias("xA"),
            F.col("selected_by_percent").cast("double").alias("ownership_pct"),
            F.col("team").alias("team_id"),
            F.col("element_type").alias("position_id"),
            F.col("status"),
        )
    )


@dp.materialized_view(
    name="gold_player_value",
    comment="Player value rankings for analytics",
)
@dp.expect("has_points", "total_points IS NOT NULL")
def gold_player_value():
    return (
        spark.read.table("silver_players")
        .filter("minutes > 0")
        .withColumn(
            "points_per_million",
            F.round(F.col("total_points") / F.col("current_price"), 2),
        )
        .withColumn(
            "points_per_90",
            F.round(F.col("total_points") / (F.col("minutes") / 90.0), 2),
        )
        .orderBy(F.desc("points_per_million"))
    )